In [4]:
import cv2
import json
from pathlib import Path

In [5]:
def calibrate_colours(video_path, frame_number=50, output_dir="colour calibration"):
    """
    Press 'r', 'g', or 'b' to select which dot colour you're currently
    clicking on. You can switch back and forth between colours as many
    times as you like (e.g. click some red, switch to green, come back
    to red later to add more samples). Press 'q' when done to save.

    Saves HSV lower/upper bounds to:
        <output_dir>/<video_name>_colour_calibration.json
    """
    video_path = Path(video_path)
    cap = cv2.VideoCapture(str(video_path))
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_number)
    ret, frame = cap.read()
    cap.release()
    if not ret:
        raise RuntimeError(f"Could not read frame {frame_number} from {video_path}")

    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

    key_to_colour = {ord("r"): "red", ord("g"): "green", ord("b"): "blue"}
    display_names = {"red": "RED", "green": "GREEN", "blue": "BLUE"}

    state = {"colour": None, "samples": {"red": [], "green": [], "blue": []}}
    margin_s = 20  # loosen saturation/value a bit vs raw clicks, to catch anti-aliased edge pixels
    margin_v = 20
    margin_h = 2

    def on_click(event, x, y, flags, param):
        if event == cv2.EVENT_LBUTTONDOWN:
            if state["colour"] is None:
                print("Press 'r', 'g', or 'b' first to pick a colour before clicking.")
                return
            h, s, v = hsv[y, x]
            state["samples"][state["colour"]].append((int(h), int(s), int(v)))
            print(f"[{state['colour']}] clicked ({x},{y}) -> HSV: ({h}, {s}, {v})  "
                  f"(total samples: {len(state['samples'][state['colour']])})")

    window = "r=red  g=green  b=blue  click to sample  q=finish"
    cv2.imshow(window, frame)
    cv2.setMouseCallback(window, on_click)

    print("Press 'r', 'g', or 'b' to start sampling a colour, click on the dot, "
          "switch colours anytime, press 'q' when finished.")

    while True:
        key = cv2.waitKey(1) & 0xFF
        if key == ord("q"):
            break
        elif key in key_to_colour:
            state["colour"] = key_to_colour[key]
            print(f"Now sampling: {display_names[state['colour']]}")

    cv2.destroyAllWindows()

    def build_range(samples, colour_name):
        if not samples:
            return None
        hs = [p[0] for p in samples]
        ss = [p[1] for p in samples]
        vs = [p[2] for p in samples]

        if colour_name == "red":
            # handle hue wraparound: treat samples as low-end or high-end red
            low_hs = [h for h in hs if h <= 90]
            high_hs = [h for h in hs if h > 90]
            ranges = []
            if low_hs:
                ranges.append([[0, max(0, min(ss) - margin_s), max(0, min(vs) - margin_v)],
                               [min(10, max(low_hs) + margin_h), 255, 255]])
            if high_hs:
                ranges.append([[max(170, min(high_hs) - margin_h), max(0, min(ss) - margin_s), max(0, min(vs) - margin_v)],
                               [180, 255, 255]])
            if not ranges:
                ranges = [[[0, max(0, min(ss) - margin_s), max(0, min(vs) - margin_v)], [10, 255, 255]],
                          [[170, max(0, min(ss) - margin_s), max(0, min(vs) - margin_v)], [180, 255, 255]]]
            return ranges
        else:
            return [[[max(0, min(hs) - margin_h), max(0, min(ss) - margin_s), max(0, min(vs) - margin_v)],
                     [max(hs) + margin_h, 255, 255]]]

    result = {}
    for colour in ("red", "green", "blue"):
        rng = build_range(state["samples"][colour], colour)
        if rng:
            result[colour] = rng
        else:
            print(f"No samples clicked for {colour}, leaving it out of the saved file.")

    out_dir = Path(output_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"{video_path.stem}_colour_calibration.json"
    out_path.write_text(json.dumps(result, indent=2))

    print(f"\nSaved colour ranges to {out_path}:")
    print(json.dumps(result, indent=2))
    return result

In [6]:
if __name__ == "__main__":
    calibrate_colours("Test_1.MOV", frame_number=50)

RuntimeError: Could not read frame 50 from Test_1.MOV